
### Imputation Redux: Imputing Missing Categorical Data

Categorical data presents a particular kind of challenge, because (as we discussed) sklearn doesn't like to deal with non-numeric data. For instance, the KNNImputer will fail if we attempt to encode categorical values.



In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder

# Set random seed for reproducibility
np.random.seed(42)

# Create synthetic data
n_samples = 100
age = np.random.randint(18, 80, n_samples)
income = np.random.randint(20000, 100000, n_samples)
education = np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n_samples)
city = np.random.choice(['New York', 'Los Angeles', 'Chicago', 'Houston', 'Miami'], n_samples)


# Create a DataFrame
df = pd.DataFrame({
    'Age': age,
    'Income': income,
    'Education': education,
    'City': city
})

# Introduce missing values
for column in df.columns:
    mask = np.random.choice([True, False], size=df.shape[0], p=[0.1, 0.9])
    df.loc[mask, column] = np.nan



# Try to use KNNImputer on the entire DataFrame
imputer = KNNImputer(n_neighbors=5)

try:
    imputed_data = imputer.fit_transform(df)
    print("\nImputed data:")
    print(imputed_data[:10])
except ValueError as e:
    print("\nError when trying to impute:")
    print(e)

There are two strategies for dealing with this; we can use SKLearn's "most frequent" method for Simple Imputation.  Alternatively (if we want to use something like a KNNImputer) we'll need to encode the data first, and then impute.



#### Using Pandas or Most Frequent Category

You can use pandas to replace nulls, using one of the methods we covered previously. Alternatively, you can use SimpleImputer with the `strategy='most_frequent'` option to impute missing values with the most frequent category in each column before one-hot encoding.


In [ ]:
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np

df_imputed = df.copy()
# Impute missing values
imp = SimpleImputer(strategy='most_frequent')
df_imputed[["Education","City"]] = imp.fit_transform(df[['Education','City']])
df_imputed


### Encoding and then imputing

If we wanted to use something more sophisticated, like a KNNImputer, we might think we try to encode first, and then impute, but there is another subtle problem here.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

le_education = OrdinalEncoder()
le_city = OrdinalEncoder()

df_numeric = df.copy()
# Fit and transform the non-null values
df_numeric['Education'] = le_education.fit_transform(df[['Education']])
df_numeric['City'] = le_city.fit_transform(df[['City']])

# Impute missing values
imputer = KNNImputer(n_neighbors=5)
imputed_data_numeric = imputer.fit_transform(df_numeric)

# Create a new DataFrame with imputed values
df_imputed = pd.DataFrame(imputed_data_numeric, columns=df.columns)
df_imputed

In [ ]:
df.info()

Here's the problem ...

In [ ]:
df_imputed[(df.Education.isna()) | (df.City.isna())]

You'll note that KNN works by calculating the _mean_ of its nearest neighbors. That doesn't make
much sense here — "Bachelor" and "PhD" don't average to anything. We *could* round the result, and
that works better than you might expect, but it's still a fudge: we're averaging category codes
whose numeric distances are arbitrary. `Master` is not "between" `Bachelor` and `PhD` in any way
the encoder knows about.

The better move is to **predict** the missing category rather than average it. The following is a
little advanced, but I've provided it here so you can refer back. For our purposes, a
`SimpleImputer` is usually sufficient.

#### A Better Approach: `IterativeImputer`

`IterativeImputer` treats imputation as a **prediction problem**. For each column with missing
values, it fits a model using the *other* columns as features and predicts what the missing
values should be. It then repeats the whole process several times (`max_iter`), each pass using
the previous pass's estimates, until the numbers settle down.

The key detail for us: **you choose the model.** Give it a `RandomForestClassifier` and it
predicts *classes* — so it can only ever return a real category, never 2.4.

##### A new import: `sklearn.experimental`

```python
from sklearn.experimental import enable_iterative_imputer
```

This line looks strange and you haven't seen anything like it before. It is not a typo and it is
not optional.

Most scikit-learn features are stable: once released, their behaviour won't change out from under
you. `IterativeImputer` is still marked **experimental**, meaning the scikit-learn developers
reserve the right to change how it works in a future version. To make sure nobody depends on it
by accident, they hid it behind this opt-in import — importing `enable_iterative_imputer` is you
saying *"I know this may change; let me use it anyway."*

Two consequences:
1. **Order matters.** The `enable_` import must come *before* you import `IterativeImputer`, or
   you get an `ImportError`.
2. **Results may shift between scikit-learn versions.** Fine for coursework and exploration; think
   twice before putting it somewhere that has to behave identically for years.

##### One parameter worth knowing: `skip_complete=True`

`IterativeImputer` works round-robin: it takes each column in turn as the thing to predict, using
the others as features. By default that includes columns with *no* missing values, which is
wasted work — and if you've handed it a classifier, it will try to "classify" `Income` into its
sixty-odd distinct values and warn you that this looks like a regression problem.

`skip_complete=True` tells it to only model columns that actually have gaps. The complete columns
are still used as *predictors*, which is what we want.

You will meet this pattern occasionally in other libraries. It's a useful signal — a maintainer
telling you exactly how much they promise.

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # must come FIRST — see above
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.ensemble import RandomForestClassifier

cat_cols = ['Education', 'City']
num_cols = ['Age', 'Income']

# 1. Encode the categories as integers, keeping NaN as NaN so the imputer can see the holes.
#    We keep the mapping so we can translate back afterwards.
codes = {c: {v: i for i, v in enumerate(sorted(df[c].dropna().unique()))} for c in cat_cols}

work = df.copy()
for c in cat_cols:
    work[c] = work[c].map(codes[c])

# 2. The classifier needs complete features to predict FROM, so fill the numeric holes first.
work[num_cols] = SimpleImputer(strategy='median').fit_transform(work[num_cols])

# 3. Predict the missing categories with a classifier.
imp = IterativeImputer(
    estimator=RandomForestClassifier(n_estimators=50, random_state=0),
    initial_strategy='most_frequent',   # what to assume on the very first pass
    skip_complete=True,                 # don't model columns that have no gaps (see note above)
    max_iter=10,
    random_state=0,
)
filled = pd.DataFrame(imp.fit_transform(work), columns=work.columns)

# 4. Translate the integers back into category names.
result = df.copy()
result[num_cols] = filled[num_cols]
for c in cat_cols:
    inv = {i: v for v, i in codes[c].items()}
    result[c] = filled[c].round().astype(int).map(inv)

print("Rows that originally had a missing category:")
result.loc[df.Education.isna() | df.City.isna()].head(8)

Every imputed value is a real category — no 2.4, and no rounding needed to get there.

#### But is it actually better?

This is the question to ask about *any* sophisticated method, and it is very easy to skip.
`IterativeImputer` is slower and more complicated than `SimpleImputer(strategy='most_frequent')`.
Is the extra machinery buying us anything?

The only way to know is to hide values we already know the answer to, impute them, and count how
many we got right — compared against the simplest thing that could possibly work. That comparison
is called a **baseline**, and you should be building one for every modelling decision you make in
this course.

In [ ]:
import numpy as np

# Build data where Education genuinely DEPENDS on Age and Income,
# then hide 20% of it so we know the right answers.
rng = np.random.RandomState(7)
N = 600
age = rng.randint(18, 80, N)
income = rng.randint(20000, 100000, N)
signal = (income / 100000) + (age / 200) + rng.normal(0, 0.10, N)
edu = np.digitize(signal, np.quantile(signal, [.35, .65, .85]))

truth_df = pd.DataFrame({'Age': age * 1.0, 'Income': income * 1.0, 'EduCode': edu * 1.0})
hidden = rng.rand(N) < 0.20
observed = truth_df.copy()
observed.loc[hidden, 'EduCode'] = np.nan
answers = truth_df.loc[hidden, 'EduCode'].values

baseline = SimpleImputer(strategy='most_frequent').fit_transform(observed)[hidden, 2]
smart = IterativeImputer(
    estimator=RandomForestClassifier(n_estimators=60, random_state=0),
    initial_strategy='most_frequent', max_iter=10,
    skip_complete=True, random_state=0
).fit_transform(observed)[hidden, 2]

print(f"most_frequent (baseline) : {np.mean(baseline == answers):.1%}")
print(f"IterativeImputer + RF    : {np.mean(smart == answers):.1%}")

Roughly double the baseline — worth the trouble here.

**But read the setup carefully: I *built* this data so that `Education` depends on `Age` and
`Income`.** That's the only reason a classifier can predict it. If a categorical column is
genuinely unrelated to everything else in your table, `IterativeImputer` cannot beat
`most_frequent` — there is no signal to find, and you will have spent CPU time to tie.

So the honest summary is: there is still no elegant, general solution for imputing categorical
data. `most_frequent` is a perfectly respectable default. Reach for something fancier when you
have reason to think the column is predictable from the others — **and then check, against a
baseline, whether it actually was.**

Finally: any of this can be packaged as a reusable transformer, so it drops straight into a
`Pipeline` alongside everything else.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin


class CategoricalIterativeImputer(BaseEstimator, TransformerMixin):
    """Impute missing categorical values by predicting them with a classifier.

    Inheriting from BaseEstimator and TransformerMixin is what makes this usable
    anywhere scikit-learn expects a transformer -- including inside a Pipeline.
    All we have to supply is `fit` and `transform`.
    """

    def __init__(self, estimator=None, max_iter=10, random_state=0):
        # scikit-learn convention: __init__ only stores arguments, it does no work.
        self.estimator = estimator
        self.max_iter = max_iter
        self.random_state = random_state

    def fit(self, X, y=None):
        self.cat_cols_ = X.select_dtypes(include=['object', 'category']).columns.tolist()
        self.num_cols_ = [c for c in X.columns if c not in self.cat_cols_]
        self.codes_ = {c: {v: i for i, v in enumerate(sorted(X[c].dropna().unique()))}
                       for c in self.cat_cols_}

        est = self.estimator or RandomForestClassifier(n_estimators=50,
                                                       random_state=self.random_state)
        self.num_imputer_ = SimpleImputer(strategy='median')
        self.imputer_ = IterativeImputer(estimator=est, initial_strategy='most_frequent',
                                         max_iter=self.max_iter, skip_complete=True,
                                         random_state=self.random_state)
        self.imputer_.fit(self._encode(X, fitting=True))
        return self

    def _encode(self, X, fitting=False):
        W = X.copy()
        for c in self.cat_cols_:
            W[c] = W[c].map(self.codes_[c])
        if self.num_cols_:
            W[self.num_cols_] = (self.num_imputer_.fit_transform(W[self.num_cols_]) if fitting
                                 else self.num_imputer_.transform(W[self.num_cols_]))
        return W

    def transform(self, X):
        filled = pd.DataFrame(self.imputer_.transform(self._encode(X)),
                              columns=X.columns, index=X.index)
        out = X.copy()
        if self.num_cols_:
            out[self.num_cols_] = filled[self.num_cols_]
        for c in self.cat_cols_:
            inv = {i: v for v, i in self.codes_[c].items()}
            out[c] = filled[c].round().astype(int).map(inv)
        return out


imputed = CategoricalIterativeImputer().fit(df).transform(df)
print("nulls remaining:", imputed.isna().sum().sum())
imputed.loc[df.Education.isna() | df.City.isna()].head()